In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense,Conv2D,Flatten,MaxPooling2D
from keras import Sequential
from keras.datasets import mnist

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [ ]:
x_train, x_test = x_train / 255.0, x_test / 255.0

In [ ]:
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

In [ ]:
model = Sequential()
model.add(Conv2D(8, kernel_size=(3,3),activation='relu', name = 'conv1', input_shape=(28,28,1)))
model.add(MaxPooling2D(pool_size=(2,2), name = 'pool1'))
model.add(Conv2D(16, kernel_size=(3, 3), activation='relu', name='conv2'))
model.add(MaxPooling2D(pool_size=(2, 2), name='pool2'))
model.add(Flatten(name='flatten'))
model.add(Dense(64, activation='relu', name='dense1'))
model.add(Dense(10, activation='softmax', name='output'))

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
print("\n--- Starting Training ---")
history = model.fit(x_train, y_train, epochs=5, validation_split=0.1, batch_size=64)

In [ ]:
print("\n--- Evaluating Model ---")
test_loss, test_acc = model.evaluate(x_test,  y_test, verbose=2)
print(f"\nFinal Test Accuracy: {test_acc*100:.2f}%")

In [ ]:
import numpy as np
import tensorflow as tf

# Load the model you just trained (Make sure the filename matches what you saved)
model_name = 'mnist_hls_model.keras' # Change to .h5 if you saved it as .h5
try:
    model = tf.keras.models.load_model(model_name)
    print(f"Successfully loaded {model_name}")
except Exception as e:
    print(f"Error loading model: {e}")

# Open a new header file to write the C++ arrays
with open('weights.h', 'w') as f:
    f.write('// Auto-generated weights for HLS MNIST CNN\n')
    f.write('#pragma once\n\n')

    # Iterate through every layer in your model
    for layer in model.layers:
        # get_weights() returns a list: [weights_array, biases_array]
        layer_weights = layer.get_weights()

        if len(layer_weights) > 0:
            print(f"Extracting weights from layer: {layer.name}")

            # Extract and flatten the multi-dimensional arrays into 1D arrays for C++
            w = layer_weights[0].flatten()
            b = layer_weights[1].flatten()

            # 1. Write the Weights
            f.write(f'const float {layer.name}_weights[{len(w)}] = {{\n')
            # Join all numbers with a comma, format to 6 decimal places
            f.write(', '.join([f"{val:.6f}" for val in w]))
            f.write('\n};\n\n')

            # 2. Write the Biases
            f.write(f'const float {layer.name}_biases[{len(b)}] = {{\n')
            f.write(', '.join([f"{val:.6f}" for val in b]))
            f.write('\n};\n\n')

print("\nSuccess! Download 'weights.h' from your Colab files.")